In [1]:
%pip install -q torch firebase-admin pandas numpy scikit-learn tqdm

Note: you may need to restart the kernel to use updated packages.


## 3) Mount Google Drive

Run this in Colab to access your model/checkpoint folder.

In [ ]:
import os

# Works in both Google Colab and local VS Code Jupyter.
IS_COLAB: bool = False
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    PROJECT_DIR: str = '/content/drive/MyDrive/BDG'
    IS_COLAB = True
    print('Runtime: Google Colab')
except ModuleNotFoundError:
    # Local fallback for VS Code notebook execution
    PROJECT_DIR = os.path.abspath('.')
    print('Runtime: Local Jupyter/VS Code (google.colab not available)')

MODEL_DIR: str = os.path.join(PROJECT_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)
print('PROJECT_DIR =', PROJECT_DIR)
print('MODEL_DIR =', MODEL_DIR)


Runtime: Local Jupyter/VS Code (google.colab not available)
PROJECT_DIR = c:\Users\raval\OneDrive\Desktop\BDG
MODEL_DIR = c:\Users\raval\OneDrive\Desktop\BDG\models


## 4) Firebase Connection

Set `SERVICE_ACCOUNT_PATH` to your uploaded JSON path in Drive.

Note: rotate leaked keys before production use.

In [ ]:
import os
import firebase_admin  # type: ignore
from firebase_admin import credentials, firestore  # type: ignore

# Try common locations so the same notebook works in Colab and local VS Code.
service_account_candidates: list[str] = [
    '/content/drive/MyDrive/BDG/firebase-adminsdk.json',
    os.path.join(PROJECT_DIR, 'firebase-adminsdk.json'),
    os.path.join(PROJECT_DIR, 'bdg_predictor', 'firebase-adminsdk.json'),
]

SERVICE_ACCOUNT_PATH: str | None = next((p for p in service_account_candidates if os.path.exists(p)), None)
if SERVICE_ACCOUNT_PATH is None:
    raise FileNotFoundError(
        'firebase-adminsdk.json not found. Place it in one of:\n' +
        '\n'.join(service_account_candidates)
    )

# Initialize Firebase app if not already initialized
try:
    firebase_admin.get_app()
except ValueError:
    cred = credentials.Certificate(SERVICE_ACCOUNT_PATH)
    firebase_admin.initialize_app(cred)

db = firestore.client()
print('Firebase connected')
print('SERVICE_ACCOUNT_PATH =', SERVICE_ACCOUNT_PATH)


Firebase connected
SERVICE_ACCOUNT_PATH = c:\Users\raval\OneDrive\Desktop\BDG\bdg_predictor\firebase-adminsdk.json


## 5) Load History Data from Firestore

In [ ]:
import pandas as pd

def load_history(limit: int = 200000) -> pd.DataFrame:
    docs = db.collection('bdg_history').order_by('ts').limit(limit).stream()
    rows: list[dict] = []
    for d in docs:
        x: dict = d.to_dict() or {}
        n = x.get('number')
        if n is None:
            continue
        rows.append({
            'period': str(x.get('period', '')),
            'number': int(n),
            'game_code': str(x.get('game_code', 'unknown')),
            'ts': x.get('ts')
        })
    df = pd.DataFrame(rows)
    if len(df):
        df = df[(df['number'] >= 0) & (df['number'] <= 9)].sort_values('ts').reset_index(drop=True)
    return df

df = load_history()
print('Rows:', len(df))
if len(df):
    print(df['game_code'].value_counts(dropna=False))
    display(df.tail(5))


Rows: 12094
game_code
unknown      11159
WinGo_30S      560
WinGo_1M       223
WinGo_3M        95
WinGo_5M        57
Name: count, dtype: int64


,period,number,game_code,ts
12089,20260318100010660,1,unknown,2026-03-18 11:00:40.739000+00:00
12090,20260318100010661,6,unknown,2026-03-18 11:01:47.732000+00:00
12091,20260318100010662,5,unknown,2026-03-18 11:02:52.570000+00:00
12092,20260318100010663,3,unknown,2026-03-18 11:03:38.874000+00:00
12093,20260318100010664,4,unknown,2026-03-18 11:04:09.334000+00:00


## 6) Train and Save LSTM (per mode)

This trains one model per game mode and saves both `latest.pt` and `best.pt` to Drive every run.

In [ ]:
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset

DEVICE: str = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE =', DEVICE)

class SeqDataset(Dataset):
    def __init__(self, numbers: list[int], seq_len: int = 30):
        self.samples: list[tuple[list[int], int]] = []
        for i in range(seq_len, len(numbers)):
            self.samples.append((numbers[i-seq_len:i], numbers[i]))
    def __len__(self) -> int:
        return len(self.samples)
    def __getitem__(self, i: int) -> tuple[torch.Tensor, torch.Tensor]:
        x, y = self.samples[i]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

class LSTMNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(10, 32)
        self.lstm = nn.LSTM(32, 256, num_layers=2, batch_first=True, dropout=0.25)
        self.drop = nn.Dropout(0.25)
        self.fc = nn.Linear(256, 10)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.emb(x)
        o, _ = self.lstm(z)
        o = self.drop(o[:, -1, :])
        return self.fc(o)

def train_mode(mode: str, seq_len: int = 30, epochs: int = 20, batch_size: int = 64, lr: float = 1e-3):
    part = df[df['game_code'] == mode].copy()
    nums = part['number'].tolist()
    if len(nums) < seq_len + 200:
        print(f'Skip {mode}: not enough rows ({len(nums)})')
        return None

    split = int(len(nums) * 0.85)
    tr, va = nums[:split], nums[split:]
    tr_ds = SeqDataset(tr, seq_len)
    va_ds = SeqDataset(va, seq_len)
    tr_dl = DataLoader(tr_ds, batch_size=batch_size, shuffle=True)
    va_dl = DataLoader(va_ds, batch_size=batch_size)

    model = LSTMNet().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()

    mode_dir = os.path.join(MODEL_DIR, mode)
    os.makedirs(mode_dir, exist_ok=True)
    best_val = 1e9

    history = []
    for ep in range(1, epochs + 1):
        model.train()
        loss_sum = 0.0
        n = 0
        for xb, yb in tr_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = model(xb)
            loss = crit(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            loss_sum += loss.item() * xb.size(0)
            n += xb.size(0)
        tr_loss = loss_sum / max(n, 1)

        model.eval()
        val_sum = 0.0
        vn = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for xb, yb in va_dl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                vloss = crit(logits, yb)
                val_sum += vloss.item() * xb.size(0)
                vn += xb.size(0)
                pred = logits.argmax(dim=1)
                correct += (pred == yb).sum().item()
                total += yb.size(0)
        val_loss = val_sum / max(vn, 1)
        val_acc = correct / max(total, 1)

        payload = {
            'epoch': ep,
            'model_state': model.state_dict(),
            'optim_state': opt.state_dict(),
            'val_loss': float(val_loss),
            'val_acc': float(val_acc),
            'mode': mode,
            'rows': len(nums),
        }

        latest_path = os.path.join(mode_dir, 'latest.pt')
        torch.save(payload, latest_path)

        if val_loss < best_val:
            best_val = val_loss
            best_path = os.path.join(mode_dir, 'best.pt')
            torch.save(payload, best_path)

        history.append({'epoch': ep, 'train_loss': float(tr_loss), 'val_loss': float(val_loss), 'val_acc': float(val_acc)})
        print(f'{mode} ep {ep:02d}: train={tr_loss:.4f} val={val_loss:.4f} acc={val_acc:.4f}')

    with open(os.path.join(mode_dir, 'history.json'), 'w') as f:
        json.dump(history, f, indent=2)

    return {'mode': mode, 'rows': len(nums), 'best_val_loss': float(best_val)}

reports: list[dict] = []
for mode in ['WinGo_30S', 'WinGo_1M', 'WinGo_3M', 'WinGo_5M']:
    r = train_mode(mode)
    if r:
        reports.append(r)

print('Training complete:', reports)


DEVICE = cpu
WinGo_30S ep 01: train=2.3006 val=2.3004 acc=0.0370
WinGo_30S ep 02: train=2.2890 val=2.3052 acc=0.0556
WinGo_30S ep 03: train=2.2749 val=2.3010 acc=0.0926
WinGo_30S ep 04: train=2.2633 val=2.3113 acc=0.1481
WinGo_30S ep 05: train=2.2354 val=2.3135 acc=0.0926
WinGo_30S ep 06: train=2.2155 val=2.3413 acc=0.0741
WinGo_30S ep 07: train=2.1837 val=2.3956 acc=0.1111
WinGo_30S ep 08: train=2.1617 val=2.3300 acc=0.1111
WinGo_30S ep 09: train=2.1354 val=2.4328 acc=0.0926
WinGo_30S ep 10: train=2.1057 val=2.4714 acc=0.1296
WinGo_30S ep 11: train=2.0369 val=2.4595 acc=0.0926
WinGo_30S ep 12: train=2.0363 val=2.5228 acc=0.1111
WinGo_30S ep 13: train=1.9912 val=2.5550 acc=0.0926
WinGo_30S ep 14: train=1.9179 val=2.6059 acc=0.1111
WinGo_30S ep 15: train=1.8560 val=2.6887 acc=0.1111
WinGo_30S ep 16: train=1.7927 val=2.7007 acc=0.0926
WinGo_30S ep 17: train=1.7086 val=2.8429 acc=0.1481
WinGo_30S ep 18: train=1.6282 val=2.8449 acc=0.1296
WinGo_30S ep 19: train=1.5573 val=3.0039 acc=0.1111

## 7) Incremental Retraining

Run this cell later to refresh models from newly collected Firebase rounds.

In [6]:
# Quick incremental retrain: shorter epochs, same save paths
inc_reports = []
for mode in ['WinGo_30S', 'WinGo_1M', 'WinGo_3M', 'WinGo_5M']:
    r = train_mode(mode, epochs=5, lr=5e-4)
    if r:
        inc_reports.append(r)
print('Incremental update complete:', inc_reports)

WinGo_30S ep 01: train=2.3032 val=2.3155 acc=0.0370
WinGo_30S ep 02: train=2.2938 val=2.3135 acc=0.0370
WinGo_30S ep 03: train=2.2869 val=2.3116 acc=0.0370
WinGo_30S ep 04: train=2.2770 val=2.3134 acc=0.0741
WinGo_30S ep 05: train=2.2696 val=2.3096 acc=0.0556
Skip WinGo_1M: not enough rows (223)
Skip WinGo_3M: not enough rows (95)
Skip WinGo_5M: not enough rows (57)
Incremental update complete: [{'mode': 'WinGo_30S', 'rows': 560, 'best_val_loss': 2.3096117973327637}]


## 8) Advanced Sequence Model (uses more detail from Firebase rows)

This model uses:
- draw number sequence
- game mode sequence
- engineered per-round features (size/color flags, period digits, rolling stats, repeat streak)
- top-1 and top-3 accuracy tracking
- class weighting for imbalanced digits
- checkpointing every epoch (`latest.pt`) and best model save (`best.pt`)

It trains one model per game mode and one global model.

In [ ]:
import os
import json
import math
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
print("Advanced trainer device:", DEVICE)

if 'MODEL_DIR' not in globals():
    MODEL_DIR: str = './models'
    os.makedirs(MODEL_DIR, exist_ok=True)
    print('MODEL_DIR not found in globals, using fallback:', MODEL_DIR)

MODE_TO_ID: dict[str, int] = {
    "WinGo_30S": 0,
    "WinGo_1M": 1,
    "WinGo_3M": 2,
    "WinGo_5M": 3,
    "unknown": 4,
}

def _safe_int_tail(s: str | None, n: int, default: int = 0) -> int:
    s = "" if s is None else str(s)
    tail = ''.join(ch for ch in s[-n:] if ch.isdigit())
    return int(tail) if tail else default

def add_engineered_features(frame: pd.DataFrame) -> pd.DataFrame:
    f = frame.copy()
    f["game_code"] = f["game_code"].fillna("unknown").astype(str)
    f["mode_id"] = f["game_code"].map(lambda x: MODE_TO_ID.get(x, MODE_TO_ID["unknown"]))
    # Basic derived flags from draw number
    f["is_big"] = (f["number"] >= 5).astype(np.float32)
    f["is_green_base"] = f["number"].isin([1, 3, 5, 7, 9]).astype(np.float32)
    f["is_red_base"] = f["number"].isin([0, 2, 4, 6, 8]).astype(np.float32)
    f["is_violet"] = f["number"].isin([0, 5]).astype(np.float32)
    # Period-derived time-like signals
    f["period_last2"] = f["period"].map(lambda x: _safe_int_tail(x, 2)).astype(np.float32)
    f["period_last3"] = f["period"].map(lambda x: _safe_int_tail(x, 3)).astype(np.float32)
    f["period_last2_norm"] = f["period_last2"] / 99.0
    f["period_last3_norm"] = f["period_last3"] / 999.0
    # Rolling trends (short + medium windows)
    f["roll_big_10"] = f["is_big"].rolling(10, min_periods=1).mean()
    f["roll_green_10"] = f["is_green_base"].rolling(10, min_periods=1).mean()
    f["roll_big_30"] = f["is_big"].rolling(30, min_periods=1).mean()
    f["roll_green_30"] = f["is_green_base"].rolling(30, min_periods=1).mean()
    # Repeat streak length
    nums = f["number"].tolist()
    streak = []
    c = 1
    for i, n in enumerate(nums):
      if i > 0 and n == nums[i - 1]:
          c += 1
      else:
          c = 1
      streak.append(c)
    f["repeat_streak"] = np.array(streak, dtype=np.float32)
    f["repeat_streak_norm"] = np.clip(f["repeat_streak"], 1, 10) / 10.0
    # Recent frequency of each current number over last 30 samples
    freq30 = []
    arr = f["number"].tolist()
    for i, n in enumerate(arr):
        start = max(0, i - 29)
        window = arr[start:i + 1]
        freq30.append(window.count(n) / max(len(window), 1))
    f["self_freq_30"] = np.array(freq30, dtype=np.float32)
    return f

FEATURE_COLS: List[str] = [
    "is_big",
    "is_green_base",
    "is_red_base",
    "is_violet",
    "period_last2_norm",
    "period_last3_norm",
    "roll_big_10",
    "roll_green_10",
    "roll_big_30",
    "roll_green_30",
    "repeat_streak_norm",
    "self_freq_30",
]

class RichSequenceDataset(Dataset):
    def __init__(self, numbers: List[int], mode_ids: List[int], feats: np.ndarray, seq_len: int = 40):
        self.samples: List[Tuple[List[int], List[int], np.ndarray, int]] = []
        n = len(numbers)
        for i in range(seq_len, n):
            x_num = numbers[i-seq_len:i]
            x_mode = mode_ids[i-seq_len:i]
            x_feat = feats[i-seq_len:i]
            y = numbers[i]
            self.samples.append((x_num, x_mode, x_feat, y))
    def __len__(self) -> int:
        return len(self.samples)
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        x_num, x_mode, x_feat, y = self.samples[idx]
        return (
            torch.tensor(x_num, dtype=torch.long),
            torch.tensor(x_mode, dtype=torch.long),
            torch.tensor(x_feat, dtype=torch.float32),
            torch.tensor(y, dtype=torch.long),
        )

class RichLSTMModel(nn.Module):
    def __init__(self, feature_dim: int, num_modes: int = 5):
        super().__init__()
        self.num_emb = nn.Embedding(10, 32)
        self.mode_emb = nn.Embedding(num_modes, 8)
        inp_dim = 32 + 8 + feature_dim
        self.lstm = nn.LSTM(inp_dim, 256, num_layers=2, batch_first=True, dropout=0.25)
        self.dropout = nn.Dropout(0.25)
        self.head = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 10),
        )
    def forward(self, x_num: torch.Tensor, x_mode: torch.Tensor, x_feat: torch.Tensor) -> torch.Tensor:
        z_num = self.num_emb(x_num)
        z_mode = self.mode_emb(x_mode)
        z = torch.cat([z_num, z_mode, x_feat], dim=-1)
        out, _ = self.lstm(z)
        out = self.dropout(out[:, -1, :])
        return self.head(out)

@dataclass
class TrainConfig:
    seq_len: int = 40
    batch_size: int = 96
    epochs: int = 25
    lr: float = 1e-3
    grad_clip: float = 1.0
    early_stop_patience: int = 6

def compute_class_weights(y_train: List[int]) -> torch.Tensor:
    counts = np.bincount(np.array(y_train, dtype=np.int64), minlength=10).astype(np.float32)
    counts[counts == 0] = 1.0
    inv = 1.0 / counts
    w = inv / inv.mean()
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)

def evaluate_model(model: nn.Module, loader: DataLoader, criterion: nn.Module) -> Dict[str, float]:
    model.eval()
    loss_sum = 0.0
    n = 0
    top1 = 0
    top3 = 0
    with torch.no_grad():
        for x_num, x_mode, x_feat, y in loader:
            x_num = x_num.to(DEVICE)
            x_mode = x_mode.to(DEVICE)
            x_feat = x_feat.to(DEVICE)
            y = y.to(DEVICE)
            logits = model(x_num, x_mode, x_feat)
            loss = criterion(logits, y)
            bs = y.size(0)
            loss_sum += loss.item() * bs
            n += bs
            pred1 = logits.argmax(dim=1)
            top1 += (pred1 == y).sum().item()
            pred3 = torch.topk(logits, k=3, dim=1).indices
            top3 += (pred3 == y.unsqueeze(1)).any(dim=1).sum().item()
    return {
        "loss": float(loss_sum / max(n, 1)),
        "top1": float(top1 / max(n, 1)),
        "top3": float(top3 / max(n, 1)),
    }


Advanced trainer device: cpu
Advanced model code loaded. Next cell: run training.


## 9) Run Advanced Training

In [8]:
# Compatibility import used by advanced trainer
from torch.utils.data import Subset

In [13]:
cfg = TrainConfig(seq_len=40, batch_size=96, epochs=30, lr=1e-3, early_stop_patience=6)
advanced_reports = run_advanced_training(df, cfg)
print(json.dumps(advanced_reports, indent=2))

[global] ep 01 | train=2.3041 val=2.3023 top1=0.0962 top3=0.3212 rows=12094 samples=12054 seq=40
[global] ep 02 | train=2.3025 val=2.3042 top1=0.0923 top3=0.3007 rows=12094 samples=12054 seq=40
[global] ep 03 | train=2.2993 val=2.3173 top1=0.0923 top3=0.2852 rows=12094 samples=12054 seq=40
[global] ep 04 | train=2.2904 val=2.3343 top1=0.1155 top3=0.3350 rows=12094 samples=12054 seq=40
[global] ep 05 | train=2.2707 val=2.5113 top1=0.1078 top3=0.3472 rows=12094 samples=12054 seq=40
[global] ep 06 | train=2.2581 val=2.3498 top1=0.1089 top3=0.3488 rows=12094 samples=12054 seq=40
[global] ep 07 | train=2.2409 val=2.4584 top1=0.1299 top3=0.3427 rows=12094 samples=12054 seq=40
[global] Early stop triggered
[WinGo_30S] ep 01 | train=2.3065 val=2.3046 top1=0.0769 top3=0.2692 rows=560 samples=520 seq=40
[WinGo_30S] ep 02 | train=2.3012 val=2.3076 top1=0.0769 top3=0.2308 rows=560 samples=520 seq=40
[WinGo_30S] ep 03 | train=2.2983 val=2.3067 top1=0.0769 top3=0.2179 rows=560 samples=520 seq=40
[Wi

## 9.5) Use Loaded Models Now (Inference)

Loads saved advanced checkpoints and predicts the next number for global and each mode using latest available sequence window.

In [10]:
import os
import json
import torch
import pandas as pd
import numpy as np

def _checkpoint_path(model_name: str) -> str:
    model_dir = os.path.join(MODEL_DIR, "advanced", model_name)
    best_path = os.path.join(model_dir, "best.pt")
    latest_path = os.path.join(model_dir, "latest.pt")
    if os.path.exists(best_path):
        return best_path
    if os.path.exists(latest_path):
        return latest_path
    raise FileNotFoundError(f"No checkpoint found for {model_name} in {model_dir}")

def _predict_from_checkpoint(df_source: pd.DataFrame, model_name: str) -> dict:
    ckpt_path = _checkpoint_path(model_name)
    payload = torch.load(ckpt_path, map_location=DEVICE)

    if "model_state" not in payload:
        return {
            "model": model_name,
            "checkpoint": ckpt_path,
            "status": payload.get("status", "checkpoint_without_model_state"),
            "rows": int(payload.get("rows", 0)),
        }

    seq_len = int(payload.get("seq_len", 40))
    frame = add_engineered_features(df_source).sort_values("ts").reset_index(drop=True)

    if model_name != "global":
        frame = frame[frame["game_code"] == model_name].reset_index(drop=True)

    if len(frame) < seq_len:
        return {
            "model": model_name,
            "checkpoint": ckpt_path,
            "status": "not_enough_rows_for_inference",
            "rows": int(len(frame)),
            "required_seq_len": seq_len,
        }

    tail = frame.tail(seq_len)
    num_arr = tail["number"].astype(int).to_numpy(dtype=np.int64)
    mode_arr = tail["mode_id"].astype(int).to_numpy(dtype=np.int64)
    feat_arr = tail[FEATURE_COLS].to_numpy(dtype=np.float32)

    x_num = torch.from_numpy(num_arr[None, :]).to(DEVICE)
    x_mode = torch.from_numpy(mode_arr[None, :]).to(DEVICE)
    x_feat = torch.from_numpy(feat_arr[None, :, :]).to(DEVICE)

    model = RichLSTMModel(feature_dim=len(FEATURE_COLS)).to(DEVICE)
    model.load_state_dict(payload["model_state"])
    model.eval()

    with torch.no_grad():
        logits = model(x_num, x_mode, x_feat)
        probs = torch.softmax(logits, dim=1).squeeze(0).cpu()
        topk = torch.topk(probs, k=3)

    top3 = [
        {"number": int(idx), "prob": float(prob)}
        for idx, prob in zip(topk.indices.tolist(), topk.values.tolist())
    ]

    return {
        "model": model_name,
        "checkpoint": ckpt_path,
        "status": "ok",
        "rows": int(len(frame)),
        "seq_len": seq_len,
        "top1_number": int(top3[0]["number"]),
        "top1_prob": float(top3[0]["prob"]),
        "top3": top3,
    }

targets = ["global", "WinGo_30S", "WinGo_1M", "WinGo_3M", "WinGo_5M"]
live_predictions = {}
for name in targets:
    try:
        live_predictions[name] = _predict_from_checkpoint(df, name)
    except Exception as e:
        live_predictions[name] = {"model": name, "status": "error", "error": str(e)}

print(json.dumps(live_predictions, indent=2))

{
  "global": {
    "model": "global",
    "checkpoint": "c:\\Users\\raval\\OneDrive\\Desktop\\BDG\\models\\advanced\\global\\best.pt",
    "status": "ok",
    "rows": 12094,
    "seq_len": 40,
    "top1_number": 0,
    "top1_prob": 0.10941318422555923,
    "top3": [
      {
        "number": 0,
        "prob": 0.10941318422555923
      },
      {
        "number": 4,
        "prob": 0.10604125261306763
      },
      {
        "number": 2,
        "prob": 0.10404808074235916
      }
    ]
  },
  "WinGo_30S": {
    "model": "WinGo_30S",
    "checkpoint": "c:\\Users\\raval\\OneDrive\\Desktop\\BDG\\models\\advanced\\WinGo_30S\\best.pt",
    "status": "ok",
    "rows": 560,
    "seq_len": 40,
    "top1_number": 0,
    "top1_prob": 0.10801080614328384,
    "top3": [
      {
        "number": 0,
        "prob": 0.10801080614328384
      },
      {
        "number": 8,
        "prob": 0.1055683046579361
      },
      {
        "number": 5,
        "prob": 0.10503743588924408
      }
    ]
 

## 10) Incremental Advanced Update (new rounds)

Use shorter training periodically after new Firebase data arrives.

In [11]:
cfg_inc = TrainConfig(seq_len=40, batch_size=96, epochs=5, lr=5e-4, early_stop_patience=3)
advanced_inc_reports = run_advanced_training(df, cfg_inc)
print(json.dumps(advanced_inc_reports, indent=2))

[global] ep 01 | train=2.3040 val=2.3011 top1=0.1150 top3=0.3278 rows=12094 samples=12054 seq=40
[global] ep 02 | train=2.3023 val=2.3010 top1=0.1095 top3=0.3074 rows=12094 samples=12054 seq=40
[global] ep 03 | train=2.2990 val=2.3059 top1=0.0995 top3=0.2968 rows=12094 samples=12054 seq=40
[global] ep 04 | train=2.2893 val=2.3343 top1=0.1128 top3=0.3134 rows=12094 samples=12054 seq=40
[global] ep 05 | train=2.2709 val=2.3294 top1=0.1244 top3=0.3400 rows=12094 samples=12054 seq=40
[global] Early stop triggered
[WinGo_30S] ep 01 | train=2.3042 val=2.3050 top1=0.0769 top3=0.2949 rows=560 samples=520 seq=40
[WinGo_30S] ep 02 | train=2.3024 val=2.3033 top1=0.0769 top3=0.3462 rows=560 samples=520 seq=40
[WinGo_30S] ep 03 | train=2.3006 val=2.3036 top1=0.0897 top3=0.3718 rows=560 samples=520 seq=40
[WinGo_30S] ep 04 | train=2.2980 val=2.3037 top1=0.1282 top3=0.3333 rows=560 samples=520 seq=40
[WinGo_30S] ep 05 | train=2.2973 val=2.3036 top1=0.1154 top3=0.3462 rows=560 samples=520 seq=40
[WinG